In [1]:
%load_ext autoreload
%autoreload 2

from mi.gpu import check_gpu_availability

check_gpu_availability()


PyTorch version 2.13.0
Apple Silicon GPU


# Tokenizer

In [2]:
from reasoning_from_scratch.qwen3 import download_qwen3_small
download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

In [3]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [4]:
# Encode
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(input_token_ids_list)

[840, 20772, 3460, 4128, 4119, 13]


In [5]:
# Decode
text = tokenizer.decode(input_token_ids_list)
print(text)

Explain large language models.


In [6]:
for i in input_token_ids_list:
    print(f"{i:5} -> {tokenizer.decode([i])}")

  840 -> Ex
20772 -> plain
 3460 ->  large
 4128 ->  language
 4119 ->  models
   13 -> .


# Pretrained Model

In [8]:
from mi.gpu import get_device
device = get_device()
print(device)

Using Apple Silicon GPU (MPS)
mps


In [10]:
# Override device
import torch
device = torch.device("cpu")

In [11]:
# Download Qwen3 model
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="downloads")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [12]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("downloads") / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

# Text Generation